# 🧠 BrainTumorXAI — 98%+ Accuracy 3-Phase Ensemble Deep Learning Training Pipeline

**Models:** DenseNet121 + InceptionV3 + EfficientNetV2S (Tri-Ensemble / Dual-Ensemble)  
**Optimization Strategy:** 3-Phase Progressive Unfreezing + Label Smoothing + Class Balancing + Test-Time Augmentation (TTA)  
**Target Accuracy:** > 98.0% Macro-F1 & Validation Accuracy  
**Compatible Platforms:** Google Colab (T4 GPU), Kaggle GPU (P100/T4), Local GPU

## 📦 Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q tensorflow keras scikit-learn matplotlib seaborn

import os, sys, gc, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121, InceptionV3, EfficientNetV2S
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus if gpus else "None (Running on CPU)")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✔ GPU Memory Growth Enabled.")
    except Exception as e:
        print("GPU config note:", e)

## ⚙️ Step 2: Auto-Detect Dataset Path & Global Configuration

In [ ]:
# Candidate paths for Kaggle, Google Colab, and Local environments
candidate_paths = [
    '/kaggle/input/datasets/bholadev58/data-source/brain-tumor-2d-dataset/image',
    '/kaggle/input/brain-tumor-2d-dataset/image',
    'datasets/image',
    'brain-tumor-2d-dataset/image',
    '../datasets/image',
    '/content/datasets/image',
    '/content/brain-tumor-2d-dataset/image'
]

DATASET_PATH = None
for path in candidate_paths:
    if os.path.exists(path) and os.path.isdir(path):
        DATASET_PATH = path
        break

if DATASET_PATH is None:
    # Fallback to current working directory datasets folder
    DATASET_PATH = 'datasets/image'
    print(f"⚠️ Warning: Auto-detect did not find candidate path. Set DATASET_PATH manually if needed: {DATASET_PATH}")
else:
    print(f"✔ Located Dataset at: {DATASET_PATH}")

# Hyperparameters for High-Accuracy Training
BATCH_SIZE   = 16         # 16 is optimal for medical CNN fine-tuning stability
VAL_SPLIT    = 0.20       # 80/20 train/validation split
SEED         = 42
NUM_CLASSES  = 4

# Epoch budget per phase
EPOCHS_P1    = 15         # Phase 1: Frozen backbone (train classifier head)
EPOCHS_P2    = 25         # Phase 2: Top-60 layers fine-tuning
EPOCHS_P3    = 20         # Phase 3: Full backbone fine-tuning (aggressive)

if os.path.exists(DATASET_PATH):
    classes = sorted([d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))])
    print("Detected Classes in Directory:", classes)

## 🔄 Step 3: Multi-Resolution Data Generators with Medical Augmentations

In [ ]:
def make_generators(target_size, augment=True):
    if augment:
        train_gen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            horizontal_flip=True,
            vertical_flip=False,
            zoom_range=0.10,
            shear_range=0.08,
            brightness_range=[0.90, 1.10],
            width_shift_range=0.08,
            height_shift_range=0.08,
            fill_mode='constant',
            cval=0,
            validation_split=VAL_SPLIT
        )
    else:
        train_gen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)

    val_gen = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)

    train_data = train_gen.flow_from_directory(
        DATASET_PATH,
        target_size=target_size,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        seed=SEED,
        shuffle=True
    )
    val_data = val_gen.flow_from_directory(
        DATASET_PATH,
        target_size=target_size,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        seed=SEED,
        shuffle=False
    )
    return train_data, val_data

print("⏳ Loading 224x224 generators (DenseNet121 & EfficientNetV2S)...")
train_224, val_224 = make_generators((224, 224))

print("\n⏳ Loading 299x299 generators (InceptionV3 Native Resolution)...")
train_299, val_299 = make_generators((299, 299))

print("\nClass mapping:", train_224.class_indices)
print(f"Total Training Samples:   {train_224.samples}")
print(f"Total Validation Samples: {val_224.samples}")

## ⚖️ Step 4: Compute Balanced Class Weights

In [ ]:
labels_train = train_224.classes
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_train),
    y=labels_train
)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Calculated Balanced Class Weights:")
for k, v in class_weight_dict.items():
    print(f"  Class {k}: {v:.4f}")

## 🏗️ Step 5: Advanced Model Architecture Builder & Callbacks

In [ ]:
def build_model(base_model, num_classes=4, dropout=0.30):
    base_model.trainable = False  # Freeze for Phase 1

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.0002))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.0002))(x)
    x = layers.Dropout(dropout)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=1e-3, amsgrad=True),
        loss=CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )
    return model

def get_callbacks(name, lr_patience=4):
    return [
        ModelCheckpoint(f'{name}_best.keras', monitor='val_accuracy',
                        save_best_only=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.4,
                          patience=lr_patience, min_lr=1e-7, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=8,
                      restore_best_weights=True, verbose=1)
    ]

print("Building DenseNet121...")
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model_densenet = build_model(base_dense)

print("Building InceptionV3...")
base_inc = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
model_inception = build_model(base_inc)

print("Building EfficientNetV2S...")
base_eff = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
model_effnet = build_model(base_eff)

print("\n✔ All 3 Backbones initialized successfully!")

## 🚀 Step 6: Phase 1 Training (Frozen Base Head Alignment)

In [ ]:
print("=" * 60)
print("▶ PHASE 1: DenseNet121 (Frozen Base, LR=1e-3)")
print("=" * 60)
h_dense_p1 = model_densenet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P1,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('densenet_p1')
)

print("\n" + "=" * 60)
print("▶ PHASE 1: InceptionV3 (Frozen Base, LR=1e-3)")
print("=" * 60)
h_inc_p1 = model_inception.fit(
    train_299, validation_data=val_299, epochs=EPOCHS_P1,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('inception_p1')
)

print("\n" + "=" * 60)
print("▶ PHASE 1: EfficientNetV2S (Frozen Base, LR=1e-3)")
print("=" * 60)
h_eff_p1 = model_effnet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P1,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('effnet_p1')
)

print("\n✔ Phase 1 Complete!")

## 🔬 Step 7: Phase 2 Fine-Tuning (Top 60 Layers Unfreeze, LR=1e-5)

In [ ]:
def unfreeze_top(model, base_model, n_layers=60, lr=1e-5):
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    for layer in base_model.layers[-n_layers:]:
        layer.trainable = True

    model.compile(
        optimizer=Adam(learning_rate=lr, amsgrad=True),
        loss=CategoricalCrossentropy(label_smoothing=0.04),
        metrics=['accuracy']
    )
    total_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
    print(f"Unfrozen last {n_layers} layers. Total Trainable Parameters: {total_trainable:,}")
    return model

print("=" * 60)
print("▶ PHASE 2: DenseNet121 Fine-Tuning")
print("=" * 60)
model_densenet = unfreeze_top(model_densenet, base_dense, n_layers=60, lr=1e-5)
h_dense_p2 = model_densenet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P2,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('densenet_p2', lr_patience=4)
)

print("\n" + "=" * 60)
print("▶ PHASE 2: InceptionV3 Fine-Tuning")
print("=" * 60)
model_inception = unfreeze_top(model_inception, base_inc, n_layers=60, lr=1e-5)
h_inc_p2 = model_inception.fit(
    train_299, validation_data=val_299, epochs=EPOCHS_P2,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('inception_p2', lr_patience=4)
)

print("\n" + "=" * 60)
print("▶ PHASE 2: EfficientNetV2S Fine-Tuning")
print("=" * 60)
model_effnet = unfreeze_top(model_effnet, base_eff, n_layers=60, lr=1e-5)
h_eff_p2 = model_effnet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P2,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('effnet_p2', lr_patience=4)
)

print("\n✔ Phase 2 Complete!")

## 🔥 Step 8: Phase 3 Aggressive Fine-Tuning (Full Unfreeze, LR=3e-6) for 98%+

In [ ]:
def full_unfreeze(model, base_model, lr=3e-6):
    base_model.trainable = True
    model.compile(
        optimizer=Adam(learning_rate=lr, amsgrad=True),
        loss=CategoricalCrossentropy(label_smoothing=0.03),
        metrics=['accuracy']
    )
    print(f"All layers unfrozen. Ultra-low LR = {lr}")
    return model

print("=" * 60)
print("▶ PHASE 3: DenseNet121 Full Unfreeze")
print("=" * 60)
model_densenet = full_unfreeze(model_densenet, base_dense, lr=3e-6)
h_dense_p3 = model_densenet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P3,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('densenet_full', lr_patience=5)
)

print("\n" + "=" * 60)
print("▶ PHASE 3: InceptionV3 Full Unfreeze")
print("=" * 60)
model_inception = full_unfreeze(model_inception, base_inc, lr=3e-6)
h_inc_p3 = model_inception.fit(
    train_299, validation_data=val_299, epochs=EPOCHS_P3,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('inception_full', lr_patience=5)
)

print("\n" + "=" * 60)
print("▶ PHASE 3: EfficientNetV2S Full Unfreeze")
print("=" * 60)
model_effnet = full_unfreeze(model_effnet, base_eff, lr=3e-6)
h_eff_p3 = model_effnet.fit(
    train_224, validation_data=val_224, epochs=EPOCHS_P3,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('effnet_full', lr_patience=5)
)

print("\n✔ Phase 3 Complete! Models deeply tuned.")

## 🧪 Step 9: Test-Time Augmentation (TTA) & Soft-Voting Ensemble Evaluation

In [ ]:
def make_tta_generator(target_size):
    tta_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        horizontal_flip=True,
        zoom_range=0.08,
        validation_split=VAL_SPLIT
    )
    return tta_datagen.flow_from_directory(
        DATASET_PATH, target_size=target_size,
        batch_size=BATCH_SIZE, class_mode='categorical',
        subset='validation', seed=None, shuffle=False
    )

def tta_predict(model, target_size, tta_steps=5):
    preds = []
    for i in range(tta_steps):
        gen = make_tta_generator(target_size)
        preds.append(model.predict(gen, verbose=0))
        print(f"  Pass {i+1}/{tta_steps} complete...")
    return np.mean(preds, axis=0)

# Load best checkpoints (fallback to p2 or p1 if stopped early)
def load_best(prefix):
    for candidate in [f'{prefix}_full_best.keras', f'{prefix}_p2_best.keras', f'{prefix}_p1_best.keras']:
        if os.path.exists(candidate):
            print(f"Loading checkpoint: {candidate}")
            return tf.keras.models.load_model(candidate)
    raise FileNotFoundError(f"No checkpoint found for {prefix}")

best_densenet  = load_best('densenet')
best_inception = load_best('inception')
best_effnet    = load_best('effnet')

# Ground truth
val_gen_clean = ImageDataGenerator(rescale=1./255, validation_split=VAL_SPLIT)
val_clean = val_gen_clean.flow_from_directory(
    DATASET_PATH, target_size=(224, 224),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', seed=SEED, shuffle=False
)
true_labels = val_clean.classes
class_names = list(val_clean.class_indices.keys())

print("\nRunning TTA Predictions on DenseNet121...")
pred_dense = tta_predict(best_densenet, (224, 224))

print("\nRunning TTA Predictions on InceptionV3...")
pred_inc = tta_predict(best_inception, (299, 299))

print("\nRunning TTA Predictions on EfficientNetV2S...")
pred_eff = tta_predict(best_effnet, (224, 224))

# Ensemble Soft-Voting (Weighted Fusion)
ensemble_probs = (0.35 * pred_dense) + (0.30 * pred_inc) + (0.35 * pred_eff)
ensemble_preds = np.argmax(ensemble_probs, axis=1)

final_acc = accuracy_score(true_labels, ensemble_preds) * 100
final_f1  = f1_score(true_labels, ensemble_preds, average='macro') * 100

print("\n" + "=" * 60)
print(f"🏆 FINAL ENSEMBLE VALIDATION ACCURACY: {final_acc:.2f}%")
print(f"🏆 FINAL ENSEMBLE MACRO F1-SCORE:     {final_f1:.2f}%")
print("=" * 60)
print("\nDetailed Classification Report:")
print(classification_report(true_labels, ensemble_preds, target_names=class_names, digits=4))

## 💾 Step 10: Save High-Accuracy Weights for Deployment

In [ ]:
output_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

# 1. Save standard .h5 weights for existing project compatibility
densenet_save_path = os.path.join(output_dir, 'densenet_best.h5')
inception_save_path = os.path.join(output_dir, 'inception_best.h5')
effnet_save_path = os.path.join(output_dir, 'effnet_best.keras')

best_densenet.save(densenet_save_path)
print(f"✔ Saved DenseNet121 model to: {densenet_save_path}")

best_inception.save(inception_save_path)
print(f"✔ Saved InceptionV3 model to:  {inception_save_path}")

best_effnet.save(effnet_save_path)
print(f"✔ Saved EfficientNetV2S to:    {effnet_save_path}")

# 2. Plot Confusion Matrix
cm = confusion_matrix(true_labels, ensemble_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Ensemble Confusion Matrix (Acc: {final_acc:.2f}%)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual True Label")
plt.tight_layout()
cm_path = os.path.join(output_dir, 'ensemble_confusion_matrix.png')
plt.savefig(cm_path, dpi=200)
plt.show()
print(f"✔ Saved Confusion Matrix plot to: {cm_path}")

print("\n🎉 ALL STEPS FINISHED SUCCESSFULLY! Download the .h5 / .keras files into your local project 'models/' folder.")